# Semana 23 — Model Serving: Latency, Throughput e Batching
**Fase 5: MLOps, Deployment e Agent-First Products** · Gabarito (notebook resolvido)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/05-Phase-5-MLOps-and-Agents/notebooks/semana_23_gabarito.ipynb)

Leitura de conceito desta semana: [Semana-23-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/05-Phase-5-MLOps-and-Agents/docs/Semana-23-Conceitos.md)

Use este gabarito **depois** de tentar o notebook de exercícios. Compare a sua solução, não copie.

> **GPU:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **T4 GPU** antes de começar.

In [ ]:
import torch, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0); print(device)
MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODELO); tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()

def cronometro():
    if device == "cuda": torch.cuda.synchronize()
    return time.perf_counter()

def preparar(perguntas):
    textos = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True) for p in perguntas]
    return tok(textos, return_tensors="pt", padding=True, add_special_tokens=False).to(device)

## Passo 1 — TTFT e tokens por segundo
Meça: (a) o tempo para gerar **1** token (aproxima o TTFT: é o prefill) e (b) o tempo para gerar `n` tokens. Derive o tempo por token de saída.

In [ ]:
@torch.no_grad()
def medir(pergunta, n=64, **kw):
    ent = preparar([pergunta]); args = dict(do_sample=False, pad_token_id=tok.pad_token_id, **kw)
    llm.generate(**ent, max_new_tokens=2, **args)                         # aquecimento
    t0 = cronometro(); llm.generate(**ent, max_new_tokens=1, min_new_tokens=1, **args); ttft = cronometro() - t0
    t0 = cronometro(); out = llm.generate(**ent, max_new_tokens=n, min_new_tokens=n, **args); total = cronometro() - t0
    gerados = out.shape[1] - ent["input_ids"].shape[1]
    return {"tokens_entrada": ent["input_ids"].shape[1], "ttft_ms": ttft * 1000, "total_ms": total * 1000,
            "tpot_ms": (total - ttft) / max(gerados - 1, 1) * 1000, "tokens_s": gerados / total}

print(pd.Series(medir("Explique o que é um banco vetorial.")).round(1))

## Passo 2 — O tamanho do prompt afeta o quê?

In [ ]:
base = "Resuma o texto a seguir em uma frase. "
linhas = [medir(base + "A política de reembolso foi revisada pelo comitê. " * rep, n=32) for rep in (1, 8, 32, 64)]
pd.DataFrame(linhas)[["tokens_entrada", "ttft_ms", "tpot_ms"]].round(1)

Qual coluna cresce com a entrada? Qual fica quase constante?

## Passo 3 — KV cache ligado e desligado
`use_cache=False` força o modelo a reprocessar todo o contexto a cada token.

In [ ]:
com = medir("Explique o que é um banco vetorial.", n=48, use_cache=True)
sem = medir("Explique o que é um banco vetorial.", n=48, use_cache=False)
print(f"com cache: {com['tokens_s']:.1f} tokens/s | sem cache: {sem['tokens_s']:.1f} tokens/s | ganho: {com['tokens_s'] / sem['tokens_s']:.1f}x")

## Passo 4 — Batching: throughput × latência
Gere para `B` prompts de uma vez e calcule o throughput total (tokens/s somando todas as sequências) e a latência do batch.

In [ ]:
perguntas = ["O que é RAG?", "Defina embedding.", "O que é LoRA?", "Explique quantização.", "O que é atenção?", "Defina token.",
             "O que é fine-tuning?", "Explique HNSW."] * 4

@torch.no_grad()
def medir_lote(B, n=32):
    ent = preparar(perguntas[:B]); args = dict(do_sample=False, pad_token_id=tok.pad_token_id, max_new_tokens=n, min_new_tokens=n)
    llm.generate(**ent, **args)
    t0 = cronometro(); llm.generate(**ent, **args); dt = cronometro() - t0
    return {"lote": B, "latencia_s": dt, "tokens_s": B * n / dt, "req_s": B / dt}

curva = pd.DataFrame([medir_lote(B) for B in (1, 2, 4, 8, 16, 32)]); print(curva.round(2))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(curva["lote"], curva["tokens_s"], "o-"); ax[0].set_xlabel("lote"); ax[0].set_ylabel("tokens/s (vazão)")
ax[1].plot(curva["lote"], curva["latencia_s"], "o-"); ax[1].set_xlabel("lote"); ax[1].set_ylabel("latência do lote (s)"); plt.show()

Na GPU, o throughput cresce quase linearmente com o batch enquanto a latência cresce pouco. (Na CPU o ganho é bem menor.)

## Passo 5 — Simulando uma fila: sem batching vs. batching estático
Simulação de eventos (não usa o modelo). Requisições chegam ao acaso a `taxa` req/s. O servidor leva `t_base + t_por_item × B` segundos para processar um batch de `B`.
- **Sem batching:** `B = 1` sempre.
- **Estático:** quando o servidor fica livre, pega até `B_max` requisições que já chegaram.

Implemente o loop do servidor e devolva a latência de cada requisição (término − chegada).

In [ ]:
def simular(taxa, B_max, n=4000, t_base=0.50, t_por_item=0.03, seed=0):
    rng = np.random.default_rng(seed); chegadas = np.cumsum(rng.exponential(1 / taxa, n))
    latencias, livre_em, i = [], 0.0, 0
    while i < n:
        inicio = max(livre_em, chegadas[i])                         # servidor livre e pelo menos uma requisição na fila
        j = i
        while j < n and chegadas[j] <= inicio and j - i < B_max:    # todas as que já chegaram, até B_max
            j += 1
        fim = inicio + t_base + t_por_item * (j - i)
        latencias.extend(fim - chegadas[i:j]); livre_em, i = fim, j
    return np.array(latencias)

for B in (1, 4, 16):
    l = simular(taxa=1.5, B_max=B); print(f"B_max={B:2d} | p50={np.percentile(l, 50):6.2f}s | p95={np.percentile(l, 95):6.2f}s")

Com `B_max = 1`, a capacidade é de `1 / 0,53 ≈ 1,9` req/s. O que acontece com a latência quando a taxa de chegada se aproxima da capacidade?

In [ ]:
taxas = np.linspace(0.3, 12, 24)
for B in (1, 4, 16):
    p95 = [np.percentile(simular(t, B, n=3000), 95) for t in taxas]
    plt.plot(taxas, p95, label=f"B_max={B}")
plt.ylim(0, 10); plt.axhline(2, ls="--", c="gray", label="SLA p95 = 2 s"); plt.xlabel("req/s"); plt.ylabel("p95 (s)"); plt.legend(); plt.show()

## Passo 6 — Dimensionamento
Para cada `B_max`, encontre a maior taxa (req/s) que uma réplica aguenta mantendo p95 ≤ 2 s. Depois calcule quantas réplicas são necessárias para um pico de 40 req/s com 40% de folga.

In [ ]:
def capacidade(B_max, sla=2.0):
    ok = [t for t in np.linspace(0.3, 30, 60) if np.percentile(simular(t, B_max, n=3000), 95) <= sla]
    return max(ok) if ok else 0.0

for B in (1, 4, 16):
    cap = capacidade(B)
    replicas = int(np.ceil(40 * 1.4 / cap)) if cap else None
    print(f"B_max={B:2d} | capacidade ≈ {cap:5.1f} req/s por réplica | réplicas para 40 req/s: {replicas}")

## Passo 7 (opcional) — vLLM
Em uma máquina com GPU dedicada (o Colab gratuito nem sempre comporta), o servidor de produção sobe com um comando e expõe uma API compatível com a da OpenAI:

```bash
pip install vllm
vllm serve Qwen/Qwen2.5-0.5B-Instruct --max-model-len 2048
# em outro terminal
curl http://localhost:8000/v1/chat/completions -H "Content-Type: application/json" \
  -d '{"model": "Qwen/Qwen2.5-0.5B-Instruct", "messages": [{"role": "user", "content": "Olá"}]}'
```

## Fechamento
Escreva o dimensionamento pedido no entregável usando os números medidos nos Passos 4 e 6.